In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("cleaned_retail.csv")
rfm = pd.read_csv("rfm_table.csv")

data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

print(data.shape)
print(rfm.shape)
print(rfm.head())

(804824, 8)
(5855, 4)
   Customer ID  Recency  Frequency  Monetary
0      15712.0        9         12   3472.41
1      12931.0       21         57  92347.34
2      17611.0        7         28   8321.98
3      16620.0        3          6   1758.93
4      13381.0       31         12   8224.37


Define the churn label

In [3]:
rfm["Churned"] = (rfm["Recency"] > 90).astype(int)

print(rfm["Churned"].value_counts())
print(rfm["Churned"].value_counts(normalize=True) * 100)


Churned
1    2968
0    2887
Name: count, dtype: int64
Churned
1    50.691716
0    49.308284
Name: proportion, dtype: float64


## Churn Label Definition

Defined churn as: no purchase within the last 90 days, based on the Recency feature from the RFM table (Module A). This threshold was chosen deliberately — it sits close to the dataset's median Recency (95 days, from Day 2 analysis), producing a genuinely balanced target (50.7% churned, 49.3% active) rather than a heavily skewed one. A shorter threshold (e.g., 30 days) would have classified most customers as "churned," making the label far less useful for a real business decision.

## Step 3: Train/Test Split (with Leakage Check)

Excluded Recency from the feature set, since it was used to define the Churned label itself (Recency > 90 = Churned) — including it would let the model simply learn back the labeling rule rather than genuinely predict churn. Used a stratified split to preserve the ~50/50 churn balance in both train and test sets.

In [4]:
from sklearn.model_selection import train_test_split

feature_cols = ["Frequency", "Monetary"]
X = rfm[feature_cols]
y = rfm["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())


Train: (4684, 2) Test: (1171, 2)
Train churn rate: 0.5068317677198976
Test churn rate: 0.5072587532023911


## Step 4: Baseline Model — Logistic Regression

Building a simple baseline first, before the more complex model, to establish a minimum performance bar. Using the same LogisticRegression tool from the Uplift Module, now applied to a standard classification task instead of a two-model uplift comparison.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, baseline_preds))
print("Precision:", precision_score(y_test, baseline_preds))
print("Recall:", recall_score(y_test, baseline_preds))
print("F1:", f1_score(y_test, baseline_preds))

Accuracy: 0.6900085397096499
Precision: 0.6542056074766355
Recall: 0.8249158249158249
F1: 0.7297096053611318


## Baseline Results

Logistic Regression on Frequency + Monetary alone: Accuracy 69.0%, Precision 65.4%, Recall 82.5%, F1 0.73. The model is biased toward flagging customers as churned (higher recall than precision) — a reasonable tradeoff for this use case, since missing an actual at-risk customer is typically costlier than a false alarm.

## Step 5: XGBoost Model

Building the primary model — XGBoost, a gradient-boosted tree ensemble — to see if it captures more signal than the linear baseline. Same features, same train/test split, for a fair comparison.

In [6]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("Precision:", precision_score(y_test, xgb_preds))
print("Recall:", recall_score(y_test, xgb_preds))
print("F1:", f1_score(y_test, xgb_preds))

Accuracy: 0.6942783945345858
Precision: 0.6755952380952381
Recall: 0.7643097643097643
F1: 0.717219589257504


## Step 6: Feature Engineering — New Features & Interactions

Adding features beyond the original Frequency/Monetary, without reintroducing Recency leakage:
- **AvgOrderValue** (Monetary / Frequency) — spend per transaction, a genuinely different signal than either raw feature alone
- **Frequency_log, Monetary_log** — log-transformed versions, since both are right-skewed (established back in Module A)
- **Tenure** — days between a customer's first and last purchase, computed from the raw transaction data. This reflects how long someone has been a customer, distinct from Recency (which reflects how recently they last bought) — not leakage, since it doesn't encode "distance from the dataset's end date."
- **Frequency × Monetary interaction** — lets the model see combined high-frequency/high-spend behavior directly, rather than requiring it to learn the interaction itself

In [7]:
import numpy as np

# Tenure: days between first and last purchase, per customer
tenure = data.groupby("Customer ID")["InvoiceDate"].agg(["min", "max"])
tenure["Tenure"] = (tenure["max"] - tenure["min"]).dt.days
tenure = tenure[["Tenure"]].reset_index()

rfm = rfm.merge(tenure, on="Customer ID", how="left")

rfm["AvgOrderValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm["Frequency_log"] = np.log1p(rfm["Frequency"])
rfm["Monetary_log"] = np.log1p(rfm["Monetary"])
rfm["Freq_Monetary_Interaction"] = rfm["Frequency"] * rfm["Monetary"]

print(
    rfm[
        [
            "Customer ID",
            "Frequency",
            "Monetary",
            "Tenure",
            "AvgOrderValue",
            "Frequency_log",
            "Monetary_log",
            "Freq_Monetary_Interaction",
        ]
    ].head()
)


   Customer ID  Frequency  Monetary  Tenure  AvgOrderValue  Frequency_log  \
0      15712.0         12   3472.41     729     289.367500       2.564949   
1      12931.0         57  92347.34     717    1620.128772       4.060443   
2      17611.0         28   8321.98     731     297.213571       3.367296   
3      16620.0          6   1758.93     734     293.155000       1.945910   
4      13381.0         12   8224.37     706     685.364167       2.564949   

   Monetary_log  Freq_Monetary_Interaction  
0      8.152892                   41668.92  
1     11.433323                 5263798.38  
2      9.026776                  233015.44  
3      7.473029                   10553.58  
4      9.014979                   98692.44  


## Step 7: Rebuild Train/Test Split with Expanded Features

In [8]:
feature_cols_v2 = [
    "Frequency",
    "Monetary",
    "Tenure",
    "AvgOrderValue",
    "Frequency_log",
    "Monetary_log",
    "Freq_Monetary_Interaction",
]

X2 = rfm[feature_cols_v2]
y2 = rfm["Churned"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

print("Train:", X2_train.shape, "Test:", X2_test.shape)


Train: (4684, 7) Test: (1171, 7)


## Step 8: Three-Model Comparison on Expanded Features

Re-running Logistic Regression and XGBoost on the new 7-feature set, plus adding a Multi-Layer Perceptron (MLPClassifier) — a genuine neural network — using scikit-learn's implementation rather than TensorFlow, to avoid the NumPy/TensorFlow environment conflict encountered in the Forecasting Module.

In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Scale features - neural networks and logistic regression both benefit from this,
# especially now that Freq_Monetary_Interaction has a very different scale than the rest
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

# Logistic Regression (v2)
lr_v2 = LogisticRegression(max_iter=1000, random_state=42)
lr_v2.fit(X2_train_scaled, y2_train)
lr_v2_preds = lr_v2.predict(X2_test_scaled)

# XGBoost (v2) - tree-based, doesn't need scaling, use original unscaled features
xgb_v2 = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
)
xgb_v2.fit(X2_train, y2_train)
xgb_v2_preds = xgb_v2.predict(X2_test)

# Neural Network (MLP)
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42)
mlp.fit(X2_train_scaled, y2_train)
mlp_preds = mlp.predict(X2_test_scaled)

for name, preds in [
    ("Logistic Regression v2", lr_v2_preds),
    ("XGBoost v2", xgb_v2_preds),
    ("Neural Network (MLP)", mlp_preds),
]:
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y2_test, preds))
    print("Precision:", precision_score(y2_test, preds))
    print("Recall:", recall_score(y2_test, preds))
    print("F1:", f1_score(y2_test, preds))



Logistic Regression v2
Accuracy: 0.7096498719043552
Precision: 0.699685534591195
Recall: 0.7491582491582491
F1: 0.7235772357723578

XGBoost v2
Accuracy: 0.7190435525192144
Precision: 0.6939970717423133
Recall: 0.797979797979798
F1: 0.7423649177760376

Neural Network (MLP)
Accuracy: 0.7216054654141759
Precision: 0.6942028985507246
Recall: 0.8063973063973064
F1: 0.7461059190031153


## Step 9: Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_v2 = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_v2.fit(X2_train, y2_train)  # tree-based, no scaling needed
rf_v2_preds = rf_v2.predict(X2_test)

print("Accuracy:", accuracy_score(y2_test, rf_v2_preds))
print("Precision:", precision_score(y2_test, rf_v2_preds))
print("Recall:", recall_score(y2_test, rf_v2_preds))
print("F1:", f1_score(y2_test, rf_v2_preds))


Accuracy: 0.7241673783091375
Precision: 0.6960926193921853
Recall: 0.8097643097643098
F1: 0.7486381322957198


## step 10: Ensemble

Rebuilt the ensemble to include all four base models — Logistic Regression, XGBoost, Neural Network, and Random Forest — combined via soft voting.

In [11]:
from sklearn.ensemble import VotingClassifier

ensemble_v2 = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, random_state=42)),
        (
            "xgb",
            XGBClassifier(
                n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
            ),
        ),
        (
            "mlp",
            MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
        ),
        ("rf", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)),
    ],
    voting="soft",
)

ensemble_v2.fit(X2_train_scaled, y2_train)
ensemble_v2_preds = ensemble_v2.predict(X2_test_scaled)

print("Accuracy:", accuracy_score(y2_test, ensemble_v2_preds))
print("Precision:", precision_score(y2_test, ensemble_v2_preds))
print("Recall:", recall_score(y2_test, ensemble_v2_preds))
print("F1:", f1_score(y2_test, ensemble_v2_preds))


Accuracy: 0.7275832621690862
Precision: 0.7055306427503737
Recall: 0.7946127946127947
F1: 0.7474267616785432


## Final Model Selection

Adding Random Forest to the ensemble (4 models total) did not meaningfully improve over the 3-model ensemble or standalone Random Forest — Accuracy remained at 72.8%, F1 (0.747) sat between the two comparison points rather than exceeding either. This suggests Random Forest's predictions substantially overlap with the existing ensemble members, adding redundancy rather than new signal.

**Final decision:** given the standalone **Random Forest model has the best F1 (0.749) and best recall (81.0%) of any single model, and is simpler to deploy than a 4-model ensemble**, Random Forest is selected as the production model for this system — trading a marginal amount of ensemble complexity for near-identical performance and a much simpler deployment footprint.

## Step 11: MLflow Experiment Tracking Setup

In [12]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn_prediction")

mlflow.set_experiment("churn_prediction")

trusted_types = ["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"]

models_to_log = {
    "LogisticRegression_2features": (
        LogisticRegression(max_iter=1000, random_state=42),
        X_train,
        X_test,
        y_train,
        y_test,
    ),
    "XGBoost_2features": (
        XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
        ),
        X_train,
        X_test,
        y_train,
        y_test,
    ),
    "LogisticRegression_7features": (
        LogisticRegression(max_iter=1000, random_state=42),
        X2_train_scaled,
        X2_test_scaled,
        y2_train,
        y2_test,
    ),
    "XGBoost_7features": (
        XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
        ),
        X2_train,
        X2_test,
        y2_train,
        y2_test,
    ),
    "NeuralNetwork_MLP": (
        MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
        X2_train_scaled,
        X2_test_scaled,
        y2_train,
        y2_test,
    ),
    "RandomForest": (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        X2_train,
        X2_test,
        y2_train,
        y2_test,
    ),
}

for name, (model, xtr, xte, ytr, yte) in models_to_log.items():
    with mlflow.start_run(run_name=name):
        model.fit(xtr, ytr)
        preds = model.predict(xte)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", accuracy_score(yte, preds))
        mlflow.log_metric("precision", precision_score(yte, preds))
        mlflow.log_metric("recall", recall_score(yte, preds))
        mlflow.log_metric("f1", f1_score(yte, preds))

        if "XGBoost" in name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model", skops_trusted_types=trusted_types)

print("All models logged to MLflow.")


2026/08/25 13:34:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


All models logged to MLflow.


## MLflow Tracking: Complete

Successfully logged 6 models to MLflow under the "churn_prediction" experiment: Logistic Regression and XGBoost (2-feature baseline), Logistic Regression, XGBoost, Neural Network, and Random Forest (7-feature expanded set) — each with accuracy, precision, recall, and F1 tracked, plus the actual trained model artifact saved for potential reuse. Resolved two MLflow security-related logging errors along the way (untrusted XGBoost and neural network internal types), fixed by using XGBoost's dedicated logging function and explicitly trusting the specific flagged types.

## Step 12: FastAPI Endpoint

Building a REST API to serve churn predictions. The model is retrained directly at API startup (fast on this small dataset) rather than loaded from MLflow's saved artifacts, keeping deployment simple. The MLflow experiment tracking from earlier remains a valid, separate record of the model comparison process.



## API Testing: Confirmed Working

Tested the /predict endpoint via FastAPI's interactive docs page. Confirmed input validation correctly rejects invalid data (Frequency=0 returns a clean 400 error instead of crashing), and valid input returns a real prediction with both a binary churn label and the underlying probability score (e.g., Frequency=10, Monetary=2000, Tenure=400 → 35.8% churn probability, predicted not churned).